# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-2/ML-week-1-FLY/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4: CTR / Engagement Opportunity Scoring.**

I'm picking this lane because it's the most concrete match to a real, repeated editorial decision: which page do we look at first when it's *getting seen* but not *getting clicked*. The starter dataset already ships everything this lane needs — impressions, clicks, CTR, and average position — without needing any warehouse tables yet, so I can validate the idea is worth pursuing before I touch anything bigger. It also naturally avoids the trap Lane 2 and Lane 3 are more exposed to (rebuilding a product decision flag or overclaiming semantic meaning) — CTR-vs-position is a comparison I can define and defend entirely from observed numbers.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Question:** Which visible pages (enough impressions, decent position) are under-capturing clicks relative to other pages at the same position, and therefore deserve a title/meta/snippet review first?

**Decision it improves:** which page a content editor opens first when they only have time to review a handful of pages this week.

**Who acts, and what do they do:** a content/SEO editor with limited review capacity. They'd open the top-ranked pages from my output and rewrite the title tag, meta description, or snippet structure — the things that influence whether someone clicks once a page is already showing up in search.

**Cost of a wrong call:** if I send the editor to a page whose CTR is actually normal for its position (a false positive), that's a wasted review — an hour or two of editor time spent rewriting something that wasn't broken. If I miss a page that's genuinely underperforming for its position (a false negative), a page that's already being shown to real searchers keeps losing clicks it could otherwise get, with no one aware there's a problem. Because review capacity is limited and finite, I'd rather rank confidently on well-supported cases (enough impressions, a real gap vs. its own position tier) than flag everything with any dip — false positives burn scarce editor time directly.

**Why data/ML helps here rather than a fixed rule:** a single global "low CTR" threshold doesn't work, because CTR naturally collapses as position gets worse — a 2% CTR is great at position 15 and terrible at position 2. The comparison has to be *within* a position tier, and ideally adjusted for other signals (content type, intent) too. That's a multi-signal, context-dependent judgment call that a plain if-statement can approximate but a model or a properly tier-adjusted score can do more reliably and more scalably across thousands of pages.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV and checking: (1) how many pages are even in the 'visible enough to matter' zone, (2) whether CTR really does vary meaningfully by position tier (justifying a tier-adjusted comparison instead of one global threshold), and (3) how many of those visible pages are sitting below their own tier's median CTR — i.e., how big is the review queue before I've built anything clever at all.


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1) The 'visible enough to matter' slice: real impressions, decent position
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)]
print(f"Visible pages (impressions_90d >= 500, position 1-20): {len(visible)} of {len(df)} total "
      f"({100*len(visible)/len(df):.1f}%)")

# 2) Does CTR really vary by position tier? (justifies tier-adjusted comparison over one flat threshold)
tier_median_ctr = visible.groupby("position_tier")["ctr"].median().sort_values(ascending=False)
print("\nMedian CTR by position tier:")
print(tier_median_ctr.round(3).to_string())

# 3) How many visible pages sit below their OWN tier's median CTR?
visible = visible.merge(tier_median_ctr.rename("tier_median_ctr"), on="position_tier")
visible["ctr_gap"] = visible["tier_median_ctr"] - visible["ctr"]
underperformers = visible[visible["ctr_gap"] > 0]

print(f"\nPages below their own position tier's median CTR: {len(underperformers)} of {len(visible)} "
      f"visible pages ({100*len(underperformers)/len(visible):.1f}%)")
print(f"Impressions sitting on those underperforming pages: {int(underperformers['impressions_90d'].sum()):,} "
      f"out of {int(visible['impressions_90d'].sum()):,} total visible-page impressions")


Visible pages (impressions_90d >= 500, position 1-20): 12023 of 30000 total (40.1%)

Median CTR by position tier:
position_tier
page_1      0.240
top_3       0.200
striking    0.170
page_3_5    0.155

Pages below their own position tier's median CTR: 5888 of 12023 visible pages (49.0%)
Impressions sitting on those underperforming pages: 51,517,205 out of 118,792,350 total visible-page impressions


## 4. Careful words: what I can and can't claim

**What I can say:** this is an *observed, decision-support* ranking. I can say a page's CTR is below the median CTR of other pages at the same position tier, in this dataset, over this 90-day window — that's a measured comparison, not a guess. I can say the resulting ranked list is *directional*: a reasonable place for a time-limited editor to start looking, backed by real numbers.

**What I can't say:** I can't say a page's low CTR is *caused* by a bad title or meta description — I have no experiment, so I can't rule out that the gap is just topic mismatch, competitor movement, or noise from low sample volume. I can't say rewriting the title *will* recover the clicks — that would need a before/after test I'm not running here. I also can't claim anything about Google's ranking algorithm, and I won't reproduce any raw query, URL, or client-identifying information in any output — everything stays at the pseudonymized, aggregated level the starter data ships at.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.